# 11 - Isolation Forest: segunda técnica para preços quebrados (P3)

Aplica o Isolation Forest (aprendizado não supervisionado) aos mesmos dados do baseline e compara com o LightGBM usando os mesmos rótulos e os mesmos erros sintéticos do notebook 10.

**Entradas:** tabelas `squad1/ml/features_precos_observacoes` (292.000 linhas) e `squad1/ml/features_precos_sku` (3.805 linhas), geradas no notebook 08.

**Saídas:** score de anomalia por observação; precision, recall e taxa de alerta contra os rótulos reais; recall nos erros sintéticos de ÷5 a x8 e falso alarme nas linhas normais; tabela de comparação com o LightGBM.

**Como rodar:** use "Executar tudo", sempre na ordem. A célula 2 instala as bibliotecas (%pip) e precisa ficar logo no começo, antes de qualquer outra.

In [0]:
%pip install python-dotenv

In [0]:
# ------------------------------------------------------------
# Configuração
# Importações, acesso ao lake (ADLS) e a função de leitura de tabelas Delta.
# Esta célula não lê nem grava dados.
# ------------------------------------------------------------
import os
from dotenv import load_dotenv
from pyspark.sql import functions as F

# Lê o arquivo .env (mesma pasta do notebook), onde ficam as credenciais do ADLS.
load_dotenv(".env")

# Endereço do lake: conta de armazenamento e pasta raiz da Squad 1.
STORAGE_ACCOUNT = "internshipdatalake"
lakehouse_base = f"abfss://squad1@{STORAGE_ACCOUNT}.dfs.core.windows.net"

# Credenciais de acesso ao ADLS (autenticação OAuth com um aplicativo do Azure).
# No Serverless o spark.conf.set de credenciais é bloqueado,
# por isso elas vão em cada leitura, pelo .options(**adls_options).
adls_options = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": os.getenv("ADLS_CLIENT_ID"),
    "fs.azure.account.oauth2.client.secret": os.getenv("ADLS_CLIENT_SECRET"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

def ler_delta_spark(caminho):
    """Lê uma tabela Delta do ADLS.

    caminho: endereço abfss:// da tabela (ex.: .../ml/features_precos_sku).
    Retorna um DataFrame do Spark.
    """
    return spark.read.format("delta").options(**adls_options).load(caminho)

In [0]:
# ------------------------------------------------------------
# Carga dos dados, separação treino / teste e colunas do Isolation Forest
# Lê as tabelas de features do notebook 08, separa treino (folds 1 a 4, linhas
# normais) e teste (fold 0, todas as linhas) e monta as colunas de entrada do
# Isolation Forest, que NÃO tem alvo: por isso o preço entra como coluna, junto
# com um preço de referência (mediana do treino por subcategoria + tamanho).
# Não treina nada e não altera nada no lake: só lê.
# ------------------------------------------------------------
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

obs_path = f"{lakehouse_base}/ml/features_precos_observacoes"  # uma linha por observação
df_obs = ler_delta_spark(obs_path)
print("features_precos_observacoes:", df_obs.count(), "linhas,", len(df_obs.columns), "colunas")

pdf = df_obs.toPandas()   # 292.000 linhas cabem tranquilamente na memória

# Treino: folds 1 a 4, só linhas "normais" (usar_no_treino == 1).
treino = pdf[(pdf["conjunto"] == "treino") & (pdf["usar_no_treino"] == 1)].copy()
# Teste: TODAS as linhas do fold 0 (normais e quebradas), porque o alerta é medido nelas.
teste_todas = pdf[pdf["conjunto"] == "teste"].copy()
teste_normais = teste_todas[teste_todas["usar_no_treino"] == 1]

# Preço de referência: mediana do TREINO por subcategoria + tamanho.
# Se a combinação não existir no treino, usa a mediana da subcategoria;
# se a subcategoria não existir, usa a mediana global.
mediana_global = treino["preco"].median()
med_subcat = treino.groupby("id_categoria")["preco"].median()
med_subcat_tam = treino.groupby(["id_categoria", "tamanho"])["preco"].median()

def adicionar_referencia(df):
    """Devolve uma cópia de df com a coluna preco_referencia.

    df: observações com id_categoria e tamanho.
    A referência vem só do treino, então não usa o preço do próprio teste.
    """
    saida = df.copy()
    ref_subcat = saida["id_categoria"].map(med_subcat).fillna(mediana_global)
    chave = pd.MultiIndex.from_frame(saida[["id_categoria", "tamanho"]])
    ref_tam = pd.Series(med_subcat_tam.reindex(chave).values, index=saida.index)
    saida["sem_combinacao"] = ref_tam.isna()          # linhas que precisaram do plano B
    saida["preco_referencia"] = ref_tam.fillna(ref_subcat)
    return saida

treino = adicionar_referencia(treino)
teste_todas = adicionar_referencia(teste_todas)
teste_normais = teste_todas[teste_todas["usar_no_treino"] == 1]

# Colunas de entrada do Isolation Forest (todas numéricas).
FEATURES_IF = ["log_preco", "log_razao_ref", "log_tamanho", "tamanho_relativo", "tamanho_rank_pct"]

def montar_features_if(df):
    """Monta as colunas de entrada do Isolation Forest a partir do preço atual de df.

    df: precisa ter preco, preco_referencia, log_tamanho, tamanho_relativo e tamanho_rank_pct.
    O preço <= 0 vira R$ 0,01 só para o log existir.
    """
    preco_pos = np.maximum(df["preco"], 0.01)
    X = pd.DataFrame(index=df.index)
    X["log_preco"] = np.log(preco_pos)
    X["log_razao_ref"] = np.log(preco_pos / df["preco_referencia"])
    X["log_tamanho"] = df["log_tamanho"]
    X["tamanho_relativo"] = df["tamanho_relativo"]
    X["tamanho_rank_pct"] = df["tamanho_rank_pct"]
    return X[FEATURES_IF]

X_treino = montar_features_if(treino)
X_teste = montar_features_if(teste_todas)

print("\nLinhas de treino (normais):", len(treino))
print("Linhas de teste (todas):", len(teste_todas), "| normais:", len(teste_normais))
print("SKUs no teste:", teste_todas["sku"].nunique())
print("Rótulos no teste: fraco", int(teste_todas["rotulo_fraco"].sum()),
      "| apoio", int(teste_todas["rotulo_apoio"].sum()))

print("\nMenor preço de referência (esperado > 0): R$", round(treino["preco_referencia"].min(), 2))
print("Linhas do teste sem combinação subcategoria + tamanho no treino:",
      int(teste_todas["sem_combinacao"].sum()))
print("Nulos nas colunas do Isolation Forest (esperado 0): treino",
      int(X_treino.isna().sum().sum()), "| teste", int(X_teste.isna().sum().sum()))

print("\n=== Colunas de entrada no treino ===")
print(X_treino.describe().round(2).to_string())

In [0]:
# ------------------------------------------------------------
# Treino do Isolation Forest e score de anomalia
# Treina nas linhas normais do treino (sem usar rótulos) e calcula o score de
# anomalia do treino e do teste. Score maior = linha mais estranha (0 a 1).
# Também mede, sem escolher limite ainda, o quanto o score separa os erros dos
# rótulos das linhas normais (AUC) e compara com uma referência simples:
# o tamanho da distância do preço à referência, |log_razao_ref|.
# Não altera nada no lake.
# ------------------------------------------------------------
from sklearn.metrics import roc_auc_score

modelo_if = IsolationForest(
    n_estimators=300,        # número de árvores de isolamento
    max_samples=256,         # linhas sorteadas por árvore (padrão do método)
    contamination="auto",    # não escolhe limite aqui; só queremos o score
    random_state=42,         # resultado repetível
    n_jobs=-1,
)
modelo_if.fit(X_treino)

# score_samples devolve valores negativos (quanto menor, mais estranho).
# Invertemos o sinal: score maior = mais anômalo.
treino["score_if"] = -modelo_if.score_samples(X_treino)
teste_todas["score_if"] = -modelo_if.score_samples(X_teste)

# Score por grupo no teste.
teste_todas["grupo"] = np.select(
    [teste_todas["rotulo_fraco"] == 1, teste_todas["rotulo_apoio"] == 1],
    ["erro (rótulo fraco)", "só rótulo de apoio"],
    default="normal",
)
print("=== Score do Isolation Forest por grupo (teste) ===")
print(teste_todas.groupby("grupo")["score_if"]
      .describe(percentiles=[0.05, 0.5, 0.95])[["count", "min", "5%", "50%", "95%", "max"]]
      .round(3).to_string())

print("\nScore nas linhas normais do treino: mediana",
      round(treino["score_if"].median(), 3),
      "| percentil 99,95:", round(treino["score_if"].quantile(0.9995), 3))

# AUC: chance de uma linha com erro ter score maior que uma normal (0,5 = acaso, 1,0 = perfeito).
ref_simples = X_teste["log_razao_ref"].abs()   # distância do preço à referência, em log
linhas_auc = []
for rotulo in ["rotulo_apoio", "rotulo_fraco"]:
    y = teste_todas[rotulo]
    linhas_auc.append({
        "rótulo": rotulo,
        "AUC Isolation Forest": roc_auc_score(y, teste_todas["score_if"]),
        "AUC |log_razao_ref| (referência simples)": roc_auc_score(y, ref_simples),
    })
print("\n=== AUC no teste ===")
print(pd.DataFrame(linhas_auc).round(4).to_string(index=False))

In [0]:
# ------------------------------------------------------------
# Limite do alerta do Isolation Forest e métricas contra os rótulos
# O limite é um percentil do score das linhas normais do TREINO (o teste não
# é usado para escolher). Para cada percentil, mede no teste: alertas, precision,
# recall e quantos alertas caem em linhas normais (sem rótulo).
# O critério combinado (falso alarme <= 0,05%) corresponde ao percentil 99,95.
# Não altera nada no lake.
# ------------------------------------------------------------
QUANTIS = [0.99, 0.995, 0.999, 0.9995]

linhas = []
for q in QUANTIS:
    limite = treino["score_if"].quantile(q)
    alerta = teste_todas["score_if"] > limite
    n_alertas = int(alerta.sum())
    d = {"percentil do treino": q, "limite (score)": limite,
         "n_alertas": n_alertas, "taxa_alerta_%": alerta.mean() * 100}
    for rotulo in ["fraco", "apoio"]:
        real = teste_todas["rotulo_" + rotulo] == 1
        acertos = int((alerta & real).sum())
        d["precision_" + rotulo] = acertos / n_alertas * 100 if n_alertas > 0 else np.nan
        d["recall_" + rotulo] = acertos / int(real.sum()) * 100
    d["alertas em linhas normais"] = int((alerta & (teste_todas["grupo"] == "normal")).sum())
    linhas.append(d)

print("=== Isolation Forest no teste (", len(teste_todas), "linhas; erros: fraco 918, apoio 1334) ===")
print(pd.DataFrame(linhas).round(3).to_string(index=False))
print("\nReferência, LightGBM v2 (k = 3): 1343 alertas, recall 100% nos dois rótulos, 9 alertas em linhas normais.")

In [0]:
# ------------------------------------------------------------
# Erros sintéticos moderados no Isolation Forest
# Mesmos cenários do notebook 10: o preço das linhas normais do teste é
# multiplicado por um fator (÷5 a x8). A % de linhas com score acima do limite é
# o recall nos cenários com erro e o falso alarme no cenário original (x1).
# Os limites são percentis do score das linhas normais do TREINO.
# Não altera nada no lake.
# ------------------------------------------------------------
FATORES = {
    "÷5": 1 / 5, "÷3": 1 / 3, "÷2": 1 / 2,
    "x1 (original)": 1,
    "x2": 2, "x3": 3, "x5": 5, "x8": 8,
}

def injetar_erro(df, fator):
    """Cópia de df com o preço multiplicado por 'fator' (2 = dobrado, 0,5 = metade, 1 = sem erro)."""
    saida = df.copy()
    saida["preco"] = saida["preco"] * fator
    return saida

limites_if = {q: treino["score_if"].quantile(q) for q in QUANTIS}

resultado = {}
for nome, fator in FATORES.items():
    df_cenario = injetar_erro(teste_normais, fator)
    score = -modelo_if.score_samples(montar_features_if(df_cenario))
    resultado[nome] = {f"IF p{q * 100:g}": (score > lim).mean() * 100
                       for q, lim in limites_if.items()}

tabela_if = pd.DataFrame(resultado).T
tabela_if.index.name = "cenário"

# Referência: LightGBM v2 (k = 3), valores do notebook 10 (teste, mesmos cenários).
lgbm_v2 = {"÷5": 57.6, "÷3": 25.9, "÷2": 9.9, "x1 (original)": 0.0,
           "x2": 0.2, "x3": 14.3, "x5": 57.9, "x8": 81.6}
tabela_if["LightGBM v2 (k=3)"] = pd.Series(lgbm_v2)

print("=== Teste, erros sintéticos: % de linhas em alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_if.round(1).to_string())

In [0]:
# ------------------------------------------------------------
# Variante IF-B e regra simples
# IF-B: Isolation Forest só com as colunas de preço (log_preco, log_razao_ref).
# Regra simples: |log_razao_ref|, a distância do preço à referência, sem modelo.
# Os limites são percentis (99,9 e 99,95) do score das linhas normais do TREINO.
# 1) Métricas contra os rótulos reais; 2) recall nos erros sintéticos.
# Não altera nada no lake.
# ------------------------------------------------------------
COLS_B = ["log_preco", "log_razao_ref"]
QUANTIS_B = [0.999, 0.9995]

modelo_if_b = IsolationForest(
    n_estimators=300, max_samples=256, contamination="auto", random_state=42, n_jobs=-1,
)
modelo_if_b.fit(X_treino[COLS_B])

def pontuar(df):
    """Score de anomalia de cada método para as linhas de df (maior = mais estranho)."""
    X = montar_features_if(df)
    return {
        "IF-B": -modelo_if_b.score_samples(X[COLS_B]),
        "Regra simples": X["log_razao_ref"].abs().to_numpy(),
    }

# Limites: percentis do score nas linhas normais do treino.
score_treino = pontuar(treino)
limites = {m: {q: np.quantile(score_treino[m], q) for q in QUANTIS_B} for m in score_treino}

# 1) Contra os rótulos reais, no teste.
score_teste = pontuar(teste_todas)
eh_normal = (teste_todas["grupo"] == "normal").to_numpy()
linhas = []
for metodo in score_teste:
    for q in QUANTIS_B:
        alerta = score_teste[metodo] > limites[metodo][q]
        n = int(alerta.sum())
        d = {"método": metodo, "percentil": q, "n_alertas": n}
        for rotulo in ["fraco", "apoio"]:
            real = (teste_todas["rotulo_" + rotulo] == 1).to_numpy()
            acertos = int((alerta & real).sum())
            d["precision_" + rotulo] = acertos / n * 100 if n > 0 else np.nan
            d["recall_" + rotulo] = acertos / int(real.sum()) * 100
        d["alertas em linhas normais"] = int((alerta & eh_normal).sum())
        linhas.append(d)
print("=== Contra os rótulos reais (teste) ===")
print(pd.DataFrame(linhas).round(1).to_string(index=False))
print("Referência, LightGBM v2 (k = 3): 1343 alertas, recall 100% nos dois rótulos, 9 alertas em linhas normais.")

# 2) Erros sintéticos no teste (mesmos cenários).
resultado = {}
for nome, fator in FATORES.items():
    sc = pontuar(injetar_erro(teste_normais, fator))
    resultado[nome] = {f"{m} p{q * 100:g}": (sc[m] > limites[m][q]).mean() * 100
                       for m in sc for q in QUANTIS_B}
tabela_b = pd.DataFrame(resultado).T
tabela_b.index.name = "cenário"
tabela_b["IF 5 colunas p99.95"] = tabela_if["IF p99.95"]
tabela_b["LightGBM v2 (k=3)"] = pd.Series(lgbm_v2)

print("\n=== Teste, erros sintéticos: % de linhas em alerta ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_b.round(1).to_string())

In [0]:
# ------------------------------------------------------------
# Comparação no mesmo nível de falso alarme
# O LightGBM v2 com k = 3 teve 72 falsos alarmes em 228.237 linhas normais do
# treino (0,032%). Aqui a regra simples e o IF-B usam o limite que dá exatamente
# esse falso alarme no treino, e comparamos o recall no teste.
# Não altera nada no lake.
# ------------------------------------------------------------
FA_TREINO = 72 / 228237                  # falso alarme do LightGBM v2 (k = 3) no treino
q_igual = 1 - FA_TREINO
limites_igual = {m: np.quantile(score_treino[m], q_igual) for m in score_treino}
print("Percentil usado:", round(q_igual * 100, 3), "| limites:",
      {m: round(float(l), 3) for m, l in limites_igual.items()})

# 1) Contra os rótulos reais.
linhas = []
for metodo in score_teste:
    alerta = score_teste[metodo] > limites_igual[metodo]
    n = int(alerta.sum())
    d = {"método": metodo, "n_alertas": n}
    for rotulo in ["fraco", "apoio"]:
        real = (teste_todas["rotulo_" + rotulo] == 1).to_numpy()
        acertos = int((alerta & real).sum())
        d["recall_" + rotulo] = acertos / int(real.sum()) * 100
    d["alertas em linhas normais"] = int((alerta & eh_normal).sum())
    linhas.append(d)
print("\n=== Contra os rótulos reais (teste), mesmo falso alarme no treino ===")
print(pd.DataFrame(linhas).round(1).to_string(index=False))
print("Referência, LightGBM v2 (k = 3): 1343 alertas, recall 100% nos dois rótulos, 9 alertas em linhas normais.")

# 2) Erros sintéticos no teste.
resultado = {}
for nome, fator in FATORES.items():
    sc = pontuar(injetar_erro(teste_normais, fator))
    resultado[nome] = {m: (sc[m] > limites_igual[m]).mean() * 100 for m in sc}
tabela_igual = pd.DataFrame(resultado).T
tabela_igual.index.name = "cenário"
tabela_igual["LightGBM v2 (k=3)"] = pd.Series(lgbm_v2)

print("\n=== Teste, erros sintéticos: % de linhas em alerta (mesmo falso alarme no treino) ===")
print("(erros: recall | x1 (original): falso alarme)\n")
print(tabela_igual.round(1).to_string())

## Registro de evoluções do Isolation Forest

| # | O que foi alterado | Antes | Depois | Ganho |
|---|---|---|---|---|
| 1 | Colunas do Isolation Forest: das 5 colunas do baseline para só as 2 de preço (log do preço e razão para o preço de referência) | Recall: 3,0% em ÷5, 2,4% em ×5 e 3,2% em ×8 (limite no percentil 99,95) | Recall: 22,1% em ÷5, 6,7% em ×5 e 13,0% em ×8 (limite com o mesmo falso alarme do LightGBM no treino, 0,032%) | +19,1, +4,3 e +9,8 pontos percentuais. Os limites são parecidos, mas não idênticos |
| 2 | Comparação com o LightGBM (mesma base de erros sintéticos e mesmo split) | Isolation Forest: 22,1%, 6,7% e 13,0%; 17 falsos alarmes | LightGBM: 57,6%, 57,9% e 81,6%; 9 falsos alarmes | O LightGBM pega pelo menos 2,5 vezes mais em ÷5, ÷3, ÷2, ×3, ×5 e ×8. Em ×2 nenhum dos dois pega (2,3% e 0,2%) |
| 3 | Regra simples como referência (mediana por subcategoria e tamanho) | LightGBM: 57,6% em ÷5 e 81,6% em ×8; 9 falsos alarmes | Regra simples: 62,4% em ÷5 e 84,1% em ×8; 13 falsos alarmes | Mesmo nível; o LightGBM ainda não superou a referência trivial |

## Conclusões da segunda técnica (Isolation Forest)

**O que foi feito:** aplicamos o Isolation Forest (300 árvores, 256 linhas por árvore, `random_state` 42) aos mesmos dados do baseline, treinado nas 228.237 linhas normais do treino, sem usar rótulos, e comparamos com o LightGBM v2 do notebook 10 (k = 3) e com uma regra simples, usando os mesmos rótulos e os mesmos erros sintéticos (preço de ÷5 a x8 nas 57.212 linhas normais do teste).

1. **Ajuste de colunas:** o Isolation Forest não tem alvo, então o preço precisa ser uma entrada. Usamos `log_preco`, `log_razao_ref` (log do preço sobre a mediana do treino por subcategoria + tamanho) e três medidas de tamanho (`log_tamanho`, `tamanho_relativo`, `tamanho_rank_pct`). As colunas categóricas ficaram de fora, porque o método as leria como números com ordem.

2. **Separação sem limite (AUC no teste):** o score do Isolation Forest tem AUC de 0,981 contra os dois rótulos, mas a referência simples |log_razao_ref| tem AUC de 1,000. Ou seja, os rótulos reais, que são erros extremos, são separados perfeitamente só pela distância do preço à referência. Por isso os rótulos reais não diferenciam as técnicas.

3. **Isolation Forest com 5 colunas:** com o limite do percentil 99,95 do treino (0,05% de falso alarme), pega 7,8% dos erros do rótulo de apoio (145 alertas, 41 em linhas normais). Nos erros sintéticos pega 3,0% dos ÷5, 2,4% dos x5 e 3,2% dos x8. Mesmo aceitando 1,3% de falso alarme (percentil 99), pega só 43% do rótulo de apoio e 14% dos x8.

4. **Variante IF-B (só `log_preco` e `log_razao_ref`):** melhora bastante em relação às 5 colunas, o que confirma que as colunas de tamanho diluíam o sinal, mas continua abaixo das outras técnicas.

5. **Regra simples:** alerta quando |log_razao_ref| passa de um limite escolhido no treino. Serve de checagem de sanidade: o que um modelo precisa superar.

6. **Comparação no mesmo falso alarme no treino (0,032%, o do LightGBM v2 com k = 3).** Recall no teste, em % de linhas em alerta:

| Cenário | IF-B | Regra simples | LightGBM v2 (k = 3) |
|---|---|---|---|
| ÷5 | 22,1 | 62,4 | 57,6 |
| ÷3 | 5,6 | 28,5 | 25,9 |
| ÷2 | 0,6 | 11,9 | 9,9 |
| x2 | 2,3 | 0,0 | 0,2 |
| x3 | 3,9 | 20,3 | 14,3 |
| x5 | 6,7 | 61,4 | 57,9 |
| x8 | 13,0 | 84,1 | 81,6 |
| Alertas em linhas normais (teste) | 17 | 13 | 9 |

Contra os rótulos reais, a regra simples e o LightGBM têm recall de 100% nos dois rótulos, e o IF-B tem 100% no fraco e 99,1% no de apoio.

7. **Conclusão:** o Isolation Forest não é a melhor técnica para este problema, porque não tem a noção de preço esperado do produto. A regra simples e o LightGBM v2 são equivalentes na prática (a regra pega de 2 a 6 pontos a mais, com 4 falsos alarmes a mais no teste). O LightGBM não mostrou, até aqui, vantagem sobre a mediana por subcategoria + tamanho, o que é coerente com o MAE (R\$ 9,44) e com o ruído natural de cerca de 35% por observação.

8. **Limitações:** os valores do LightGBM v2 foram copiados das saídas do notebook 10, não recalculados aqui. Os erros sintéticos não substituem rótulos reais de erros moderados. O falso alarme é um teto, pois as linhas normais podem ter erros sem rótulo. A regra simples usa um limite único para todas as categorias, enquanto o LightGBM v2 usa a MAD por categoria raiz. Testamos só uma variante do Isolation Forest, definida antes de ver os resultados.

9. **Próximos passos:** discutir com o Geovany o que isso significa para o argumento do LightGBM; testar um limite por categoria na regra simples e por subcategoria no LightGBM; procurar novas colunas que tragam informação além do preço de referência.